In [4]:
# RECOVERY CELL (self-contained). Runtime died? Run only this cell, then continue
# from your last completed step. It repeats the install, launch, and health-poll
# so you do not have to scroll. Nothing here depends on earlier cells having run.
import os, sys, time, signal, subprocess, urllib.request, urllib.error

RECOVERY_MODEL = "Qwen/Qwen2.5-1.5B-Instruct"
RECOVERY_PORT = 8000
RECOVERY_LOG = "/content/server.log"

# Pins mirrored from ../../../PINS.md (keep in sync; PINS.md wins).
_R_TRANSFORMERS = "4.46.*"   # PINS.md: mandatory beside vLLM, every fresh runtime
_R_ACCELERATE = "1.1.*"
_R_NEED_AWQ = False          # set True on day 4+ if your locked model is AWQ
_R_VLLM = "0.6.*"; _R_HTTPX = "0.27.*"; _R_OPENAI = "1.54.*"

# If a day added --quantization awq or the tool-call flags, add them here too so
# recovery brings the server back the way the lab needs it. Default is the plain
# serving config from canon.
RECOVERY_ARGS = {
    "--model": RECOVERY_MODEL,
    "--dtype": "half",
    "--max-model-len": "4096",
    "--gpu-memory-utilization": "0.85",
    "--port": str(RECOVERY_PORT),
}

# 1) kill any leftover server holding the port
subprocess.run(["pkill", "-f", "vllm.entrypoints.openai.api_server"],
               check=False)
time.sleep(2)

# 2) reinstall pins (fresh runtime has nothing)
subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                f"vllm=={_R_VLLM}", f"transformers=={_R_TRANSFORMERS}",
                f"accelerate=={_R_ACCELERATE}", f"httpx=={_R_HTTPX}",
                f"openai=={_R_OPENAI}"]
               + (["autoawq==0.2.9"] if _R_NEED_AWQ else []),
               check=True)
print("pins reinstalled")

# 3) relaunch the server in the background
_r_cmd = [sys.executable, "-m", "vllm.entrypoints.openai.api_server"]
for k, v in RECOVERY_ARGS.items():
    _r_cmd += [k] if v is None else [k, str(v)]
_r_logf = open(RECOVERY_LOG, "wb")
server = subprocess.Popen(_r_cmd, stdout=_r_logf, stderr=subprocess.STDOUT,
                          start_new_session=True)
print(f"relaunched server pid {server.pid}, logging to {RECOVERY_LOG}")

# 4) re-poll health (first load re-downloads the model; hence 300s)
_deadline = time.time() + 300
while time.time() < _deadline:
    try:
        with urllib.request.urlopen(
                f"http://localhost:{RECOVERY_PORT}/v1/models", timeout=5) as r:
            if r.status == 200:
                print("RECOVERED: server healthy. continue from your last step.")
                break
    except (urllib.error.URLError, ConnectionError, OSError):
        pass
    time.sleep(3)
else:
    print("recovery timed out. last 30 log lines:")
    try:
        with open(RECOVERY_LOG, errors="replace") as fh:
            print("".join(fh.readlines()[-30:]))
    except FileNotFoundError:
        print("(no log file)")
    print("if it keeps timing out: switch to the Kaggle fallback in the shared "
          "README, or rotate to another team Colab account.")

CalledProcessError: Command '['/usr/bin/python3', '-m', 'pip', 'install', '-q', 'vllm==0.6.*', 'transformers==4.46.*', 'accelerate==1.1.*', 'httpx==0.27.*', 'openai==1.54.*']' returned non-zero exit status 1.

In [5]:
!/content/venv/bin/python -m pip install \
    "vllm==0.6.*" \
    "transformers==4.46.*" \
    "accelerate==1.1.*" \
    "httpx==0.27.*" \
    "openai==1.54.*"

In [6]:
import os, subprocess, time, urllib.request, urllib.error

# إيقاف أي عمليات سابقة
subprocess.run(["pkill", "-f", "vllm.entrypoints.openai.api_server"], check=False)
time.sleep(2)

cmd = [
    "/content/venv/bin/python", "-m", "vllm.entrypoints.openai.api_server",
    "--model", "Qwen/Qwen2.5-1.5B-Instruct",
    "--dtype", "half",
    "--max-model-len", "4096",
    "--gpu-memory-utilization", "0.85",
    "--port", "8000"
]

log_file = open("/content/server.log", "wb")
server = subprocess.Popen(cmd, stdout=log_file, stderr=subprocess.STDOUT, start_new_session=True)
print(f"vLLM server started with PID: {server.pid}")

# فحص جاهزية السيرفر (Health Poll)
print("Waiting for server to become healthy (this takes 3-5 mins)...")
deadline = time.time() + 300
while time.time() < deadline:
    try:
        with urllib.request.urlopen("http://localhost:8000/v1/models", timeout=5) as response:
            if response.status == 200:
                print("SUCCESS: vLLM server is healthy and ready!")
                break
    except (urllib.error.URLError, ConnectionError, OSError):
        pass
    time.sleep(5)
else:
    print("Timeout! Printing last 30 lines of /content/server.log:")
    with open("/content/server.log", "r") as f:
        print("".join(f.readlines()[-30:]))

vLLM server started with PID: 12443
Waiting for server to become healthy (this takes 3-5 mins)...
SUCCESS: vLLM server is healthy and ready!


In [ ]:
# 1. Install python3.10 and venv support
!sudo apt-get update -y
!sudo apt-get install python3.10 python3.10-venv python3.10-dev -y
# 2. Create an isolated virtual environment
!python3.10 -m venv /content/venv
# 3. Upgrade pip inside the virtual environment
!/content/venv/bin/python -m pip install --upgrade pip
# 4. Install the required serving pins
!/content/venv/bin/python -m pip install \
    "vllm==0.6.*" \
    "transformers==4.46.*" \
    "accelerate==1.1.*" \
    "httpx==0.27.*" \
    "openai==1.54.*"
print("Virtual environment ready with vLLM installed!")

Get:1 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease [3,632 B]
Hit:2 https://cli.github.com/packages stable InRelease
Get:3 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  InRelease [1,578 B]
Get:4 http://security.ubuntu.com/ubuntu jammy-security InRelease [129 kB]
Get:5 https://r2u.stat.illinois.edu/ubuntu jammy InRelease [6,555 B]
Hit:6 http://archive.ubuntu.com/ubuntu jammy InRelease
Get:7 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  Packages [2,915 kB]
Get:8 http://archive.ubuntu.com/ubuntu jammy-updates InRelease [128 kB]
Hit:9 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease
Get:10 https://r2u.stat.illinois.edu/ubuntu jammy/main all Packages [10.8 MB]
Hit:11 https://ppa.launchpadcontent.net/graphics-drivers/ppa/ubuntu jammy InRelease
Hit:12 https://ppa.launchpadcontent.net/ubuntugis/ppa/ubuntu jammy InRelease
Get:13 http://archive.ubuntu.com/ubuntu jammy-backports InRelease [127

In [7]:
from openai import OpenAI
client = OpenAI(base_url="http://localhost:8000/v1", api_key="not-needed")
r = client.chat.completions.create(
    model="Qwen/Qwen2.5-1.5B-Instruct",
    messages=[{"role": "user", "content": "In one sentence, what is a GPU?"}],
)
print(r.choices[0].message.content)

A GPU, or Graphics Processing Unit, is a specialized processor designed to accelerate computations involved in rendering graphics and video content on electronic devices.


In [8]:
from google.colab import files
uploaded = files.upload()   # pick baselines.json
import json
baseline = json.load(open("baselines.json"))
print("baseline batch tokens/s:", baseline["batch"])

Saving baselines.json to baselines.json
baseline batch tokens/s: {'1': 28.2, '4': 41.6, '8': 81.1}


In [9]:
# Async A/B client for Lab W3D3 (engine swap).
# Paste the whole file as one Colab cell (after the vLLM server is healthy), then
# call run_sweep(...) as the day-3 README shows. It fires N concurrent chat
# completions per level with httpx + asyncio, excludes a warm-up round, and
# reports aggregate tokens/s at each concurrency level.
#
# It talks to the OpenAI-compatible /v1 endpoint, so the same client works
# against any team's service. No secrets: the local vLLM server needs no key.

import asyncio
import time

import httpx

# A fixed prompt set so every run measures the same work. Varied lengths, no
# duplicates. Requests cycle through this list.
FIXED_PROMPTS = [
    "In one sentence, what is a GPU?",
    "List three reasons decode is memory-bound.",
    "Explain the KV cache to a new ops engineer in two sentences.",
    "What does continuous batching change versus static batching?",
    "Give a one-line definition of tokens per second.",
    "Why does a longer prompt increase time to first token?",
    "Name two things quantisation trades away for smaller memory.",
    "Summarise what an inference server does in three short bullets.",
]

# Output lengths per request, cycled in order. This list is IDENTICAL to Monday's
# QUEUE in the day-2 lab, and it has to stay that way: the A/B is only honest if
# both engines are asked for exactly the same work. 24 requests, 18 that want 32
# tokens and 6 that want 256, so a long request is always in flight alongside
# short ones.
#
# The mixed lengths are the entire point. Ask every request for the same number
# of tokens and there is no straggler, static batching pays no tax, and
# continuous batching has nothing to win back. You would measure a flat speedup
# across concurrency and conclude, wrongly, that continuous batching does not
# scale.
QUEUE = [32, 32, 32, 256] * 6

# Fallback when a caller does not pass a length.
MAX_TOKENS = 128
# Warm-up requests per level, dropped from the timing.
WARMUP = 4


async def _one_request(client, base_url, model, prompt, max_tokens=MAX_TOKENS):
    """Fire one chat completion, return the count of completion tokens."""
    payload = {
        "model": model,
        "messages": [{"role": "user", "content": prompt}],
        "max_tokens": max_tokens,
        "temperature": 0.0,
        "stream": False,
    }
    r = await client.post(f"{base_url}/chat/completions", json=payload)
    r.raise_for_status()
    body = r.json()
    usage = body.get("usage", {})
    # completion_tokens is what the server generated; fall back to counting.
    # Accounting note vs Monday: static_queue counted REQUESTED tokens, which
    # equals generated there (greedy decode runs to the cap). vLLM can stop at
    # EOS short of the cap, so counting usage is the honest number for it -
    # any bias this introduces runs AGAINST vLLM, never for it.
    ct = usage.get("completion_tokens")
    if ct is None:
        ct = len(body["choices"][0]["message"]["content"].split())
    return ct


async def _run_level(client, base_url, model, prompts, concurrency, total_requests):
    """Run total_requests requests, at most `concurrency` in flight at once."""
    sem = asyncio.Semaphore(concurrency)
    counts = []

    async def guarded(prompt, max_tokens):
        async with sem:
            return await _one_request(client, base_url, model, prompt, max_tokens)

    # Each request carries its own output length from QUEUE, so the workload
    # matches Monday's static-batching baseline request for request.
    tasks = [asyncio.create_task(guarded(prompts[i % len(prompts)],
                                         QUEUE[i % len(QUEUE)]))
             for i in range(total_requests)]
    t0 = time.time()
    for coro in asyncio.as_completed(tasks):
        counts.append(await coro)
    dt = time.time() - t0
    total_tokens = sum(counts)
    return {
        "concurrency": concurrency,
        "requests": total_requests,
        "tokens_per_s": round(total_tokens / dt, 1),
        "wall_s": round(dt, 3),
    }


async def run_sweep(base_url, model, prompts=FIXED_PROMPTS,
                    concurrencies=(1, 4, 8), requests_per_level=24):
    """Sweep the concurrency levels; return a list of per-level result dicts.

    A warm-up round runs first and is discarded so model-load and cache-warm
    cost stays out of the measured numbers.
    """
    results = []
    async with httpx.AsyncClient(timeout=120.0) as client:
        # warm-up: fire WARMUP requests, ignore timing
        await asyncio.gather(*[
            _one_request(client, base_url, model, prompts[i % len(prompts)])
            for i in range(WARMUP)
        ])
        for c in concurrencies:
            level = await _run_level(client, base_url, model, prompts, c,
                                     requests_per_level)
            print("level:", level)
            results.append(level)
    return results


prompts = FIXED_PROMPTS            # defined in ab_client.py
vllm_measured = await run_sweep(
    base_url="http://localhost:8000/v1",
    model="Qwen/Qwen2.5-1.5B-Instruct",
    prompts=prompts,
    concurrencies=[1, 4, 8],
)
for level in vllm_measured:
    print(level)

level: {'concurrency': 1, 'requests': 24, 'tokens_per_s': 58.8, 'wall_s': 23.639}
level: {'concurrency': 4, 'requests': 24, 'tokens_per_s': 169.6, 'wall_s': 8.19}
level: {'concurrency': 8, 'requests': 24, 'tokens_per_s': 229.8, 'wall_s': 6.045}
{'concurrency': 1, 'requests': 24, 'tokens_per_s': 58.8, 'wall_s': 23.639}
{'concurrency': 4, 'requests': 24, 'tokens_per_s': 169.6, 'wall_s': 8.19}
{'concurrency': 8, 'requests': 24, 'tokens_per_s': 229.8, 'wall_s': 6.045}


In [10]:
import json

def tokps_at(level_list, c):
    return next(x["tokens_per_s"] for x in level_list if x["concurrency"] == c)

vllm_by_c = {x["concurrency"]: x["tokens_per_s"] for x in vllm_measured}
# Monday's static batching at 1/4/8 is the baseline curve
base_by_c = {int(k): v for k, v in baseline["batch"].items()}

speedup = {c: round(vllm_by_c[c] / base_by_c[c], 2)
           for c in vllm_by_c if c in base_by_c}

report = {
    "baseline": base_by_c,               # from Monday's baselines.json
    "vllm": vllm_by_c,                    # measured today
    "speedup_by_concurrency": speedup,
    "predicted_speedup": None,           # <- put your prediction-card number here
}
with open("ab_report.json", "w") as f:
    json.dump(report, f, indent=2)
print(json.dumps(report, indent=2))

{
  "baseline": {
    "1": 28.2,
    "4": 41.6,
    "8": 81.1
  },
  "vllm": {
    "1": 58.8,
    "4": 169.6,
    "8": 229.8
  },
  "speedup_by_concurrency": {
    "1": 2.09,
    "4": 4.08,
    "8": 2.83
  },
  "predicted_speedup": null
}


In [11]:
static_scaling = base_by_c[8] / base_by_c[1]
vllm_scaling   = vllm_by_c[8] / vllm_by_c[1]
print(f"static batching scales {static_scaling:.2f}x, vLLM scales {vllm_scaling:.2f}x")
print(f"continuous batching is worth {vllm_scaling / static_scaling:.2f}x of scaling")

static batching scales 2.88x, vLLM scales 3.91x
continuous batching is worth 1.36x of scaling


In [13]:
import os, signal, time, urllib.request, urllib.error

# Cell: clean shutdown.
# Terminate the server process group and confirm port 8000 is free again. Run
# this between labs, or before relaunching with different flags. Killing only the
# parent pid can leave a child holding the port; killpg kills the whole group the
# launch cell created with start_new_session=True.
def shutdown_server(proc=None, port=RECOVERY_PORT):
    try:
        proc = server if proc is None else proc
        os.killpg(os.getpgid(proc.pid), signal.SIGTERM)
        print(f"sent SIGTERM to process group of pid {proc.pid}")
    except (ProcessLookupError, NameError):
        print("no server process to kill")
    # give it a moment, then confirm the port is free
    time.sleep(3)
    try:
        with urllib.request.urlopen(f"http://localhost:{port}/v1/models", timeout=2):
            print(f"WARNING: port {port} still answering; something is still up")
    except (urllib.error.URLError, ConnectionError, OSError):
        print(f"port {port} is free")